# Phase 3 — Baseline Model

**Goal:** establish a reliable CV score baseline before any tuning. We try three models in order of complexity — Logistic Regression → Random Forest → XGBoost — and compare their cross-validated accuracy.

**Input:** `X_train`, `y_train`, `X_test` produced by `02_feature_engineering.ipynb`.  
**Output:** a submission CSV ready to upload to Kaggle.

---

## 1. Environment

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

DATA    = Path('../data/raw')
SUBMISSIONS = Path('../submissions')
SUBMISSIONS.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')

print(f'train: {train.shape}   test: {test.shape}')

## 2. Feature engineering

Identical pipeline to `02_feature_engineering.ipynb` — reproduced here so this notebook is self-contained and runnable independently.

In [ ]:
def extract_title(name_series):
    return name_series.str.split(',', n=1).str[1].str.strip().str.split('.', n=1).str[0]

TITLE_MAP = {'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master'}

df = pd.concat([train, test]).reset_index(drop=True)

# Title
df['Title'] = extract_title(df['Name']).map(TITLE_MAP).fillna('Rare')

# Fare imputation
df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))

# Sex encoding
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Age imputation via regression
title_dummies_age = pd.get_dummies(df['Title']).drop(columns='Mr').astype(int)
features_age = pd.concat([df['Pclass'], df['Sex'], df['Fare'], title_dummies_age], axis=1)
known   = features_age[df['Age'].notna()]
unknown = features_age[df['Age'].isna()]
age_model = LinearRegression()
age_model.fit(known, df.loc[df['Age'].notna(), 'Age'])
df.loc[df['Age'].isna(), 'Age'] = age_model.predict(unknown).clip(0.5, 80)

# Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['is_solo']    = (df['FamilySize'] == 1).astype(int)

# Embarked
df['Embarked'] = df['Embarked'].fillna('S')

# Dummies — cast to int, deck excluded (too sparse, hurts generalisation)
title_dummies    = pd.get_dummies(df['Title'],    prefix='Title').drop(columns='Title_Mr').astype(int)
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Emb').drop(columns='Emb_S').astype(int)

# Assemble feature matrix
FEATURE_COLS = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'is_solo']
df_model = pd.concat([df[FEATURE_COLS], title_dummies, embarked_dummies], axis=1)

train_mask = df['Survived'].notna()
X_train = df_model[train_mask].reset_index(drop=True)
y_train = df.loc[train_mask, 'Survived'].astype(int).reset_index(drop=True)
X_test  = df_model[~train_mask].reset_index(drop=True)

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'Missing in X_train: {X_train.isna().sum().sum()}  X_test: {X_test.isna().sum().sum()}')
print(f'Any bool columns: {any(X_train.dtypes == bool)}')

## 3. Cross-validation setup

**StratifiedKFold** preserves the class balance (survived/not survived) in each fold — important because the dataset is imbalanced (~38% survived). Plain KFold doesn't guarantee this.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_score(model, X, y):
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f'  Folds: {scores.round(4)}')
    print(f'  Mean:  {scores.mean():.4f}  Std: {scores.std():.4f}')
    return scores.mean()

## 4. Logistic Regression

Linear decision boundary. Requires standardised features (StandardScaler) so that large-scale features like Fare don't dominate small-scale ones like is_solo.

In [ ]:
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42)),
])

print('Logistic Regression:')
lr_score = cv_score(lr, X_train, y_train)

## 5. Random Forest

Ensemble of decision trees. No scaling needed — trees split on thresholds, not distances. Handles interactions automatically.

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)

print('Random Forest:')
rf_score = cv_score(rf, X_train, y_train)

## 6. XGBoost

Gradient boosted trees — builds sequentially, each tree correcting the errors of the previous. Generally the strongest on tabular data.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)

print('XGBoost:')
xgb_score = cv_score(xgb, X_train, y_train)

## 7. Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'CV Accuracy': [lr_score, rf_score, xgb_score],
}).sort_values('CV Accuracy', ascending=False)

print(results.to_string(index=False))
print(f'\nGender baseline (LB): 0.76555')
print(f'Previous best (LB):   0.77272')

## 8. Generate submission

Use the best CV model to generate predictions on the test set.

In [ ]:
best_model = xgb
best_name  = 'xgboost_baseline'

best_model.fit(X_train, y_train)
predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived':    predictions.astype(int),
})

from datetime import datetime
filename = SUBMISSIONS / f'{datetime.now().strftime("%Y-%m-%d_%H%M")}_{best_name}.csv'
submission.to_csv(filename, index=False)
print(f'Saved: {filename}')
print(f'Predictions: {submission.Survived.value_counts().to_dict()}')
submission.head()

## 9. Submit to Kaggle\n\nUncomment and run once you are ready to submit. Include the CV score in the message for your own records.

In [ ]:
# import subprocess
# result = subprocess.run([
#     'kaggle', 'competitions', 'submit',
#     '-c', 'titanic',
#     '-f', str(filename),
#     '-m', f'XGBoost baseline — CV {xgb_score:.4f}',
# ], capture_output=True, text=True)
# print(result.stdout or result.stderr)

---

## 10. Findings & lessons learned

**Results:**

| Model | CV Accuracy | LB Score |
|---|---|---|
| XGBoost (100 trees, no deck) | 0.8361 | 0.77700 |
| XGBoost (500 trees, with deck) | 0.8395 | 0.76000 |
| Gender baseline | — | 0.76555 |
| 2021 best (target) | — | 0.78229 |

**Key lessons:**

1. **Sparse one-hot features cause overfitting.** Deck dummies (77% unknown → 7 sparse columns) inflated CV by ~0.004 and tanked LB. The model memorised deck patterns from ~250 passengers instead of generalising. Removing them brought CV down slightly but LB up significantly.

2. **CV–LB gap is a diagnostic tool.** A gap >0.05 means something is wrong — pipeline bug, leakage, or overfitting. Don't submit blindly; investigate the gap first.

3. **Fewer trees can generalise better.** On 891 samples, 500 XGBoost trees is too many. 100 trees gave lower CV variance and better LB.

4. **Always cast `get_dummies` to int.** Pandas 2.x returns bool dtype — XGBoost mishandles it silently, producing near-random predictions with no error message.

**Next (Phase 4):** hyperparameter tuning — `learning_rate`, `max_depth`, `min_child_weight`, regularisation — to push past the 0.782 target.